# SDN on ME4OH full-field OFAM3 SST data

Experiment 4: Using SDN only to produce reconstructions

In [1]:
# Standard imports
import json
import os

# Third-party imports
import numpy as np
import torch
from sklearn.preprocessing import MinMaxScaler

# Local imports
import models  # https://github.com/Jan-Williams/pyshred/blob/main/models.py 
from Data.preprocess_sst_ofam3_shred import (
    load_files,
    split_ordered_data,
)
from helper_functions.saving import JsonEncoder

np.random.seed(42)

## SHRED preprocessing

In [2]:
exp_index = 4
test_lags = [2, 52]

# Experiment 4.1:
num_sensors = 5

# Experiment 4.2:
# num_sensors = 1059


In [3]:
# Load and reformat data
data, dates, lats, longs = load_files()

Loading data: 100%|██████████| 1878/1878 [00:10<00:00, 184.08it/s]


In [4]:
print("Preprocessing data...")
train_data, val_data, test_data, test_dates = split_ordered_data(data, dates)

# Normalise data
sc = MinMaxScaler()
sc = sc.fit(train_data)  # Computes min and max from training data only (prevent data leakage)
train_transformed = sc.transform(train_data)  
val_transformed = sc.transform(val_data)
test_transformed = sc.transform(test_data)

Preprocessing data...


In [5]:
class SDNDataset(torch.utils.data.Dataset):
    """TBC"""
    def __init__(self, X, Y):  # noqa: D107
        self.X = X
        self.Y = Y
        self.len = X.shape[0]
        
    def __getitem__(self, index):  # noqa: D105
        return self.X[index], self.Y[index]
    
    def __len__(self):  # noqa: D105
        return self.len

In [6]:
def create_sdn_datasets(data, sensor_locs, test_lags=None):
    """TBC"""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    if test_lags is None:
        # Not matching test dataset to SHRED test dataset
        data_in = np.array([data[i, sensor_locs] for i in range(len(data))])
        data_in = torch.tensor(data_in, dtype=torch.float32).to(device)
        data_out = torch.tensor(data, dtype=torch.float32).to(device)

    else:
        # Want only to test on the end point for each lag so can directly compare SDN and SHRED performance
        data_in = np.array([data[i+test_lags, sensor_locs] for i in range(len(data)-test_lags)])
        data_in = torch.tensor(data_in, dtype=torch.float32).to(device)
        ### -1 to have output be at the same time as final sensor measurements
        data_out = torch.tensor(data[np.array(range(len(data)-test_lags)) + test_lags - 1], dtype=torch.float32).to(device)

    dataset = SDNDataset(data_in, data_out)

    return dataset

In [7]:
# Build dataset
sensor_locations = np.random.choice(data.shape[1], size=num_sensors, replace=False)
sensor_coords=(longs[sensor_locations], lats[sensor_locations])

train_dataset = create_sdn_datasets(train_transformed, sensor_locations)
val_dataset = create_sdn_datasets(val_transformed, sensor_locations)
test_dataset = create_sdn_datasets(test_transformed, sensor_locations)

In [8]:
# Generate test set to match SHRED test set
# (use final sensor results for what each sequence would be if using SHRED)
test_dataset_match_shred = {}
test_lag_dates = {}
for lags in test_lags:
    test_dataset_match_shred[lags] = create_sdn_datasets(test_transformed, sensor_locations, test_lags=lags)
    test_lag_dates[lags] = test_dates[np.array(range(len(test_dates)-lags)) + lags - 1]

## Train + test SDN model

In [9]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
sdn_model = models.SDN(num_sensors, len(lats)).to(device)
validation_errors = models.fit(sdn_model, train_dataset, val_dataset, batch_size=64,
                               num_epochs=1000, lr=1e-3, verbose=True, patience=5)

torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
Training epoch 1
Error tensor(0.3079)
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 59063])
64
torch.Size([64, 5

In [10]:
test_recons = sc.inverse_transform(sdn_model(test_dataset.X).detach().cpu().numpy())
test_ground_truth = sc.inverse_transform(test_dataset.Y.detach().cpu().numpy())
print('Test Reconstruction Error: ')
print(np.linalg.norm(test_recons - test_ground_truth) / np.linalg.norm(test_ground_truth))

Test Reconstruction Error: 
0.034833234


In [11]:
test_recons_match_shred = {}
ground_truth_match_shred = {}
for lags, dataset in test_dataset_match_shred.items():
    test_recon_lags = sc.inverse_transform(sdn_model(dataset.X).detach().cpu().numpy())
    ground_truth_lags = sc.inverse_transform(dataset.Y.detach().cpu().numpy())
    print('Test Reconstruction Error: ')
    print(np.linalg.norm(test_recon_lags - ground_truth_lags)/ np.linalg.norm(ground_truth_lags))
    test_recons_match_shred[lags] = test_recon_lags
    ground_truth_match_shred[lags] = ground_truth_lags

Test Reconstruction Error: 
0.03657526
Test Reconstruction Error: 
0.036474895


## Save results

In [12]:
# Make experiment folder
directory = f"Reconstructions/Static/Exp{exp_index}/"
os.makedirs(directory, exist_ok = True)

In [13]:
np.save(f"Reconstructions/Static/Exp{exp_index}/recons_n{num_sensors}", test_recons)
np.save(f"Reconstructions/Static/Exp{exp_index}/truth_n{num_sensors}", test_ground_truth)

for lags in test_lags:
    np.save(f"Reconstructions/Static/Exp{exp_index}/recons_n{num_sensors}_l{lags}", test_recons_match_shred[lags])
    np.save(f"Reconstructions/Static/Exp{exp_index}/truth_n{num_sensors}_l{lags}", ground_truth_match_shred[lags])

In [14]:
metadata = {
    "sensor_locations": list(sensor_locations),
    "sensor_coords": list(sensor_coords),
    "test_dates": list(test_dates)
    }

In [ ]:
with open(f"Reconstructions/Static/Exp{exp_index}/metadata_n{num_sensors}.json",
          'w', encoding='utf-8') as f:
    json.dump(metadata, f, cls=JsonEncoder)

# Save metadata for matching lags testing
for lags in test_lags:
    metadata["test_lag_dates"] = test_lag_dates[lags]
    with open(f"Reconstructions/Static/Exp{exp_index}/metadata_n{num_sensors}_l{lags}.json",
          'w', encoding='utf-8') as f:
      json.dump(metadata, f, cls=JsonEncoder)

<_io.TextIOWrapper name='Reconstructions/Static/Exp4/metadata_n5_l2.json' mode='w' encoding='utf-8'>
<_io.TextIOWrapper name='Reconstructions/Static/Exp4/metadata_n5_l52.json' mode='w' encoding='utf-8'>
